# OASIS-1 MRI — exploração + Dataset de treino

Os dados estão em `oasis_cross-sectional_disc1/`. Cada paciente (`OAS1_XXXX_MR1`) tem um
volume 3D processado `*_t88_masked_gfc.img` (cérebro registrado no atlas, sem crânio),
formato **Analyze 7.5 big-endian**, shape **176 x 208 x 176** `int16`.

Aqui a gente:
1. lê os volumes só com **numpy** (sem nibabel);
2. imprime vários exemplos das imagens;
3. usa a classe `OASISDataset` (no arquivo `oasis_dataset.py`) que transforma
   **todas as fatias** de MRI em amostras de treino, cada uma com **imagem**, **altura**
   (índice da fatia) e **número do paciente**.

In [ ]:
import os, re, glob, struct
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

ROOT = "oasis_cross-sectional_disc1"

# leitor Analyze big-endian (mesmo do oasis_dataset.py)
_DTYPE = {2: ">u1", 4: ">i2", 8: ">i4", 16: ">f4", 64: ">f8"}
def read_analyze(img_path):
    with open(img_path[:-4] + ".hdr", "rb") as f:
        hdr = f.read()
    ndim = struct.unpack(">h", hdr[40:42])[0]
    dims = struct.unpack(">8h", hdr[40:56])[1:1 + ndim]
    vol = np.fromfile(img_path, dtype=_DTYPE[struct.unpack(">h", hdr[70:72])[0]]).astype(np.float32)
    vol = vol.reshape(dims[::-1]).transpose(*range(len(dims))[::-1])
    return np.squeeze(vol)

## 1. O que tem no dataset

In [ ]:
volumes = sorted(glob.glob(os.path.join(ROOT, "**", "*t88_masked_gfc.img"), recursive=True))
print(f"pacientes / volumes encontrados: {len(volumes)}\n")
for p in volumes[:5]:
    pid = re.search(r"OAS1_\d+_MR\d+", p).group(0)
    print(pid, "->", p)
print("...")

In [ ]:
# cada paciente tem um .txt com idade, sexo, CDR (0 = normal, >0 = demência)...
def read_meta(pid):
    txt = os.path.join(ROOT, "disc1", pid, pid + ".txt")
    meta = {}
    for line in open(txt):
        if ":" in line:
            k, _, v = line.partition(":")
            meta[k.strip()] = v.strip()
    return meta

print(f"{'PACIENTE':16}{'IDADE':>6}{'SEXO':>8}{'CDR':>5}{'MMSE':>6}")
for p in volumes[:8]:
    pid = re.search(r"OAS1_\d+_MR\d+", p).group(0)
    m = read_meta(pid)
    print(f"{pid:16}{m.get('AGE',''):>6}{m.get('M/F',''):>8}{m.get('CDR',''):>5}{m.get('MMSE',''):>6}")

## 2. Vários prints das imagens de MRI

In [ ]:
# um paciente nos 3 planos anatômicos. shape = (X=sagital 176, Y=coronal 208, Z=axial 176)
vol = read_analyze(volumes[0])
pid = re.search(r"OAS1_\d+_MR\d+", volumes[0]).group(0)
print(pid, "shape:", vol.shape)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(np.rot90(vol[88, :, :]), cmap="gray"); ax[0].set_title("sagital  x=88")
ax[1].imshow(np.rot90(vol[:, 104, :]), cmap="gray"); ax[1].set_title("coronal  y=104")
ax[2].imshow(np.rot90(vol[:, :, 88]),  cmap="gray"); ax[2].set_title("axial  z=88")
for a in ax: a.axis("off")
plt.suptitle(pid); plt.tight_layout(); plt.show()

In [ ]:
# muitas fatias axiais ("alturas") do mesmo paciente, de baixo pra cima
alturas = range(30, 150, 10)
fig, ax = plt.subplots(2, 6, figsize=(15, 5))
for a, z in zip(ax.ravel(), alturas):
    a.imshow(np.rot90(vol[:, :, z]), cmap="gray")
    a.set_title(f"altura z={z}"); a.axis("off")
plt.suptitle(f"{pid} — fatias axiais"); plt.tight_layout(); plt.show()

In [ ]:
# a mesma altura (z=88) em pacientes diferentes
fig, ax = plt.subplots(2, 6, figsize=(15, 5))
for a, p in zip(ax.ravel(), volumes[:12]):
    v = read_analyze(p)
    pid_i = re.search(r"OAS1_\d+", p).group(0)
    a.imshow(np.rot90(v[:, :, 88]), cmap="gray")
    a.set_title(pid_i, fontsize=9); a.axis("off")
plt.suptitle("axial z=88 — vários pacientes"); plt.tight_layout(); plt.show()

## 3. Dataset de treino: todas as fatias

A classe `OASISDataset` está no arquivo **`oasis_dataset.py`** (para você importar depois
num script de treino). Cada amostra é um dicionário com `image`, `altura` e `patient`.

In [ ]:
from oasis_dataset import OASISDataset

ds = OASISDataset(ROOT, plane="axial")   # varre os volumes e monta o índice de fatias
print(f"pacientes: {len(ds.volumes)}  |  fatias de treino: {len(ds)}")

s = ds[len(ds) // 2]
print("\numa amostra:")
print("  image  :", tuple(s['image'].shape), s['image'].dtype)
print("  altura :", s['altura'])
print("  patient:", s['patient'], f"({s['patient_id']})")

In [ ]:
# um batch do DataLoader (como no treino de verdade)
loader = DataLoader(ds, batch_size=12, shuffle=True)
batch = next(iter(loader))
print("image :", batch['image'].shape)   # [B, 1, H, W]
print("altura:", batch['altura'].tolist())
print("patient:", batch['patient'].tolist())

fig, ax = plt.subplots(2, 6, figsize=(15, 5))
for a, img, z, p in zip(ax.ravel(), batch['image'], batch['altura'], batch['patient']):
    a.imshow(np.rot90(img[0].numpy()), cmap="gray")
    a.set_title(f"pac {int(p)}  z={int(z)}", fontsize=9); a.axis("off")
plt.suptitle("batch de treino"); plt.tight_layout(); plt.show()

In [ ]:
# salvar o índice pronto num arquivo -> recarrega rápido depois, sem re-escanear
ds.save("oasis_index.pt")
print("salvo em oasis_index.pt")

ds2 = OASISDataset.load("oasis_index.pt")
print("recarregado:", len(ds2), "fatias")